## Generate map displaying road snapped points within h3 cells (of specified resolution)

In [ ]:
import h3
from h3 import LatLngPoly, LatLngMultiPoly
import json
from shapely.geometry import shape
from shapely.ops import unary_union
from sqlalchemy import create_engine, text
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
import folium
from config import DATABASE_URL

resolution = 5

engine = create_engine(DATABASE_URL)


def draw_conus_polygon():
    with open("datasets/conus-states.json", "r") as f:
        state_geo = json.load(f)

    states = [shape(f["geometry"]) for f in state_geo["features"]]
    conus = unary_union(states)
    return conus


def convert_polygon_to_h3(geom):
    if geom.geom_type == "Polygon":
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in geom.interiors
        ]
        return LatLngPoly(exterior, holes)

    elif geom.geom_type == "MultiPolygon":
        return LatLngMultiPoly(*(convert_polygon_to_h3(p) for p in geom.geoms))


def initialize_cell_centers_dataframe(cells):
    rows = [
        (cell, *h3.cell_to_latlng(cell))
        for cell in cells
    ]

    df = pd.DataFrame(rows, columns=["CellID", "Latitude", "Longitude"])

    df["Latitude"] = df["Latitude"].round(6)
    df["Longitude"] = df["Longitude"].round(6)
    return df


def get_road_snapped_point(points, batch_size=500):
    if isinstance(points, tuple) and len(points) == 2 and not isinstance(points[0], (tuple, list)):
        points = [points]

    if not points:
        return []

    normalized_points = []
    for point in points:
        if len(point) != 2:
            raise ValueError("Each point must be a (lon, lat) pair")

        lon, lat = point
        normalized_points.append((float(lon), float(lat)))

    results = []

    with engine.connect() as conn:
        for start in range(0, len(normalized_points), batch_size):
            batch = normalized_points[start:start + batch_size]
            value_sql = ", ".join(
                f"(:id_{i}, :lon_{i}, :lat_{i})"
                for i in range(len(batch))
            )
            params = {}
            for i, (lon, lat) in enumerate(batch):
                params[f"id_{i}"] = start + i
                params[f"lon_{i}"] = lon
                params[f"lat_{i}"] = lat

            sql = text(f"""
                WITH input_points(id, lon, lat) AS (
                    VALUES {value_sql}
                )
                SELECT
                    input.id,
                    ST_Y(snapped.geom_4326) AS lat,
                    ST_X(snapped.geom_4326) AS lon
                FROM (
                    SELECT
                        id,
                        ST_Transform(
                            ST_SetSRID(ST_Point(lon, lat), 4326),
                            3857
                        ) AS geom
                    FROM input_points
                ) AS input
                LEFT JOIN LATERAL (
                    SELECT
                        ST_Transform(ST_ClosestPoint(way, input.geom), 4326) AS geom_4326
                    FROM planet_osm_roads
                    WHERE highway IS NOT NULL
                    ORDER BY way <-> input.geom
                    LIMIT 1
                ) snapped ON true
                ORDER BY input.id
            """)

            batch_rows = conn.execute(sql, params).fetchall()
            results.extend(
                (round(row[1], 6), round(row[2], 6))
                for row in batch_rows
            )

    return results


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return round(R * c)


def add_road_snapped_data_to_dataframe(df):
    snapped_points = get_road_snapped_point(list(zip(df["Longitude"], df["Latitude"])))

    if len(snapped_points) != len(df):
        raise ValueError("Expected one snapped point per input row")

    df[["RSLatitude", "RSLongitude"]] = pd.DataFrame(
        snapped_points,
        columns=["RSLatitude", "RSLongitude"]
    )

    df["RSDistance"] = df.apply(
        lambda row: haversine(
            row["Latitude"],
            row["Longitude"],
            row["RSLatitude"],
            row["RSLongitude"],
        ),
        axis=1,
    )

    df["RSCellID"] = df.apply(
        lambda row: h3.latlng_to_cell(
            row["RSLatitude"],
            row["RSLongitude"],
            resolution,
        ),
        axis=1,
    )

    df["ValidSnap"] = df["CellID"] == df["RSCellID"]
    return df


conus = draw_conus_polygon()
h3_conus = convert_polygon_to_h3(conus)
h3_cells = h3.polygon_to_cells_experimental(
    h3shape=h3_conus,
    res=resolution,
    contain="overlap",
)
df = initialize_cell_centers_dataframe(h3_cells)
df = add_road_snapped_data_to_dataframe(df)


In [ ]:
# Plot data

m = folium.Map(location=(45, -115), zoom_start=7)

geojson_features = []

for row in df.itertuples(index=False):
    color = "green" if row.ValidSnap else "red"

    hexagon = [[lon, lat] for lat, lon in h3.cell_to_boundary(row.CellID)]

    geojson_features.append(
        {
            "type": "Feature",
            "properties": {"kind": "hexagon", "color": "black", "weight": 0.5},
            "geometry": {"type": "Polygon", "coordinates": [hexagon]},
        }
    )

    geojson_features.append(
        {
            "type": "Feature",
            "properties": {"kind": "line", "color": color, "weight": 2},
            "geometry": {
                "type": "LineString",
                "coordinates": [
                    [row.Longitude, row.Latitude],
                    [row.RSLongitude, row.RSLatitude],
                ],
            },
        }
    )

    folium.CircleMarker(
        location=[row.RSLatitude, row.RSLongitude],
        radius=2,
        fill=True,
        fill_opacity=1,
        color=color,
    ).add_to(m)

    folium.CircleMarker(
        location=[row.Latitude, row.Longitude],
        radius=2,
        fill=True,
        fill_opacity=1,
        color=color,
    ).add_to(m)

folium.GeoJson(
    {
        "type": "FeatureCollection",
        "features": geojson_features,
    },
    style_function=lambda feature: {
        "color": feature["properties"]["color"],
        "weight": feature["properties"]["weight"],
        "fill": False,
        "opacity": 0.8,
    },
).add_to(m)
m

In [ ]:
m.save(f"html/road_snapped_points_res{resolution}.html")

## 